In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

In [3]:
from funcoes_monitoramento import *
from funcoes_psi import *

In [4]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

In [5]:
RATING_GROUP_ORDER = {
    "A+": "A",
    "A": "A",
    "B+": "A",
    "B": "A",
    "C": "B",
    "D+": "C",
    "D": "D",
    "E": "E",
    "E-BVS": "E",
}

RATING_DS_POL_ORDER = {
    "A+":"1.A+",
    "A":"2.A",
    "B+":"3.B+",
    "B":"4.B",
    "C":"5.C",
    "D+":"6.D+",
    "D":"7.D",
    "E":"8.E",
    "E-BVS":"9.E-BVS",
}

RATING_DS_POL_COLOR_MAP = {
    "1.A+":   "#22D3EE",  # best
    "2.A":    "#38BDF8",
    "3.B+":   "#2DD4BF",
    "4.B":    "#6EE7B7",
    "5.C":    "#93C5FD",
    "6.D+":   "#A5B4FC",
    "7.D":    "#818CF8",
    "8.E":    "#475569",
    "9.E-BVS": "#1E293B",
}

RATING_DS_POL_TEXT_COLOR_MAP = {
    "1.A+":    "black",
    "2.A":     "black",
    "3.B+":    "black",
    "4.B":     "black",
    "5.C":     "black",
    "6.D+":    "white",
    "7.D":     "white",
    "8.E":     "white",
    "9.E-BVS": "white",
}

## Base Funil

In [6]:
df_raw = pd.read_csv(ANALYTICS_DIR / "df_funil_blend4.csv", low_memory=False)
df_raw["requested_at"] = pd.to_datetime(df_raw["requested_at"])

mask = df_raw["bureau_nm_ajust"] == "BLEND4"

df_raw["rating_score_group"] = df_raw["rating_score_ds"]
df_raw.loc[mask, "rating_score_group"] = (
    df_raw.loc[mask, "rating_score_ds"].replace(RATING_GROUP_ORDER)
)

df_raw["rating_score_ds"] = df_raw["rating_score_ds"]

df_raw.loc[mask, "rating_score_ds"] = (
    df_raw.loc[mask, "rating_score_ds"].replace(RATING_DS_POL_ORDER)
)

df_raw["modeloBlend_group"] = np.where(
    df_raw["is_fallback"] == 1,
    "Fallback",
    df_raw["modeloBlend"],
)

# mask_blend3_e = (
#     (df_raw["bureau_nm_ajust"] == "BLEND3")
#     & (df_raw["rating_score_ds"] == "8.E")
# )
# df_raw.loc[mask_blend3_e, "pre_analysis_result"] = "REPROVAR"

for col in ["iniciada_at", "enviada_at", "activated_at", "cancelled_at"]:
    if col in df_raw.columns:
        df_raw[col] = pd.to_datetime(df_raw[col], errors="coerce")

# Flags padronizadas do funil blend
df = prepare_blend_funnel_columns(df_raw)
df = prepare_week_columns(df, "requested_at")

# Produção real: evita duplicar contrato entre BLEND4 (simulado) e BLEND3_3 (prod)
# df_prod = df[df[MODEL_COL] == df["bureau_nm_ajust"]].copy()

print(f"Volume total (simulação): {len(df):,}")
print(f"Contratos únicos: {df['contract_id'].nunique():,}")
# print(f"Volume produção: {len(df_prod):,}")
print(f"Período: {df['requested_at'].min()} → {df['requested_at'].max()}")
# df.head()

Volume total (simulação): 114,307
Contratos únicos: 114,307
Período: 2026-07-01 00:00:00 → 2026-07-28 00:00:00


In [9]:
df_uniprop = df[df["qtd_proponentes"] == 1].copy()

df_multprop = df[df["qtd_proponentes"] >= 2].copy()

In [13]:
df_uniprop.groupby("modeloBlend_group")["rating_score_group"].value_counts()

modeloBlend_group     rating_score_group
BLEND3_3              E                     41010
                      C                     21460
                      B                     18774
                      A                     10621
                      D                      1383
                      N/I                     138
BLEND_4               E                      5370
                      A                      4930
                      D                      1706
                      C                       758
                      B                       732
                      N/I                      40
BLEND_REGRESSAO_2026  E                       467
                      B                       230
                      C                       220
                      A                       159
                      D                       137
Fallback              E                      1232
                      D                       135
         

In [14]:
df_blend4_uniprop = df_uniprop[df_uniprop["modeloBlend_group"] == "BLEND_4"].copy()
df_blend4_uniprop

,contract_id,dt_lead,requested_at,iniciada_at,enviada_at,activated_at,cancelled_at,dt_saida,tipo_contrato,product_nm,...,modeloBlend_group,is_elegivel,is_iniciada,is_enviada,is_aprovada,is_ativada,week_start,year_week,year,week_of_year
2,4421469,2026-07-25,2026-07-25,NaT,NaT,NaT,2026-07-25,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
20,4417797,2026-07-24,2026-07-24,NaT,NaT,NaT,2026-07-24,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
21,4417360,2026-07-24,2026-07-24,NaT,NaT,NaT,2026-07-24,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
22,4420100,2026-07-24,2026-07-24,NaT,NaT,NaT,2026-07-24,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
26,4416278,2026-07-24,2026-07-24,NaT,NaT,NaT,2026-07-24,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114097,4373837,2026-07-15,2026-07-15,2026-07-15,2026-07-15,2026-07-15,NaT,NaN,PF,Smart Plus,...,BLEND_4,1,1,1,1,1,2026-07-12,2026-07-12,2026,29
114125,4374387,2026-07-15,2026-07-15,NaT,NaT,NaT,NaT,NaN,PF,NaN,...,BLEND_4,1,0,0,0,0,2026-07-12,2026-07-12,2026,29
114206,4320974,2026-07-01,2026-07-01,NaT,NaT,NaT,2026-07-20,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-06-28,2026-06-28,2026,27
114288,4331022,2026-07-03,2026-07-03,NaT,NaT,NaT,NaT,NaN,PF,NaN,...,BLEND_4,1,0,0,0,0,2026-06-28,2026-06-28,2026,27


In [17]:
df_blend4_uniprop.describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])[["person_deemed_income"]]

,person_deemed_income
count,13536.000000
mean,5879.603576
min,0.000000
1%,0.000000
5%,0.000000
10%,0.000000
25%,1644.000000
50%,2603.000000
75%,5274.500000
90%,10891.500000


In [22]:
df_blend4_uniprop

,contract_id,dt_lead,requested_at,iniciada_at,enviada_at,activated_at,cancelled_at,dt_saida,tipo_contrato,product_nm,...,modeloBlend_group,is_elegivel,is_iniciada,is_enviada,is_aprovada,is_ativada,week_start,year_week,year,week_of_year
2,4421469,2026-07-25,2026-07-25,NaT,NaT,NaT,2026-07-25,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
20,4417797,2026-07-24,2026-07-24,NaT,NaT,NaT,2026-07-24,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
21,4417360,2026-07-24,2026-07-24,NaT,NaT,NaT,2026-07-24,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
22,4420100,2026-07-24,2026-07-24,NaT,NaT,NaT,2026-07-24,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
26,4416278,2026-07-24,2026-07-24,NaT,NaT,NaT,2026-07-24,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-07-19,2026-07-19,2026,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114097,4373837,2026-07-15,2026-07-15,2026-07-15,2026-07-15,2026-07-15,NaT,NaN,PF,Smart Plus,...,BLEND_4,1,1,1,1,1,2026-07-12,2026-07-12,2026,29
114125,4374387,2026-07-15,2026-07-15,NaT,NaT,NaT,NaT,NaN,PF,NaN,...,BLEND_4,1,0,0,0,0,2026-07-12,2026-07-12,2026,29
114206,4320974,2026-07-01,2026-07-01,NaT,NaT,NaT,2026-07-20,NaN,PF,NaN,...,BLEND_4,0,0,0,0,0,2026-06-28,2026-06-28,2026,27
114288,4331022,2026-07-03,2026-07-03,NaT,NaT,NaT,NaT,NaN,PF,NaN,...,BLEND_4,1,0,0,0,0,2026-06-28,2026-06-28,2026,27


In [25]:
df_blend4_uniprop[df_blend4_uniprop["person_deemed_income"]==0].groupby(["is_elegivel", "is_iniciada", "is_enviada", "is_aprovada", "is_ativada"], dropna=False).size()

is_elegivel  is_iniciada  is_enviada  is_aprovada  is_ativada
0            0            0           0            0             1546
1            0            0           0            0               21
             1            0           0            0                5
dtype: int64